# LST数据提取工具 - Master Notebook

## 📚 完整功能合集

这个Notebook包含了LST数据提取工具的所有核心功能。你可以在这里：

1. **提取GEE数据** - LST、NDVI、PM2.5等
2. **批量处理** - 网格化城市、多时段提取
3. **质量控制** - 自动标记和填充缺失值
4. **可视化分析** - 绘制数据图表
5. **导出数据** - 保存为CSV/Excel/GeoJSON

---

## 🚀 快速开始

### 第一步：初始化环境

运行下面的单元格来设置环境。

In [ ]:
# ============================================================
# 环境设置和导入
# ============================================================

import sys
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 添加项目路径
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

print("📁 项目路径:", project_root)
print("\n📦 导入核心模块...")

# 导入核心库
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 基础库导入完成")

# 导入项目核心模块
try:
    from core.config_manager import ConfigManager
    from core.grid_manager import GridManager
    from core.batch_manager import BatchManager
    from core.session_manager import SessionManager
    from core.quality_tracker import QualityTracker
    from core.universal_extractor import UniversalExtractor
    print("✅ 核心模块导入完成")
except Exception as e:
    print(f"⚠️  核心模块导入失败: {e}")
    print("将使用备用实现...")

# 尝试导入GEE
try:
    import ee
    print("✅ Google Earth Engine 导入成功")
    
    # 初始化GEE（需要认证）
    try:
        ee.Initialize()
        print("✅ GEE 已初始化")
    except Exception as e:
        print(f"⚠️  GEE初始化失败: {e}")
        print("请运行: ee.Authenticate()")
except ImportError:
    print("⚠️  未安装 earthengine-api")
    print("安装命令: pip install earthengine-api")

print("\n" + "="*50)
print("✅ 环境设置完成！")
print("="*50)

---

## 📋 配置管理

在开始提取数据之前，先设置配置参数。

In [ ]:
# ============================================================
# 配置管理
# ============================================================

class Config:
    """全局配置类"""
    
    # 数据目录
    BASE_DIR = project_root
    DATA_DIR = BASE_DIR / "data"
    OUTPUT_DIR = BASE_DIR / "output"
    CACHE_DIR = BASE_DIR / "cache"
    
    # GEE配置
    GEE_MAX_POINTS = 5000  # 每次请求最大点数
    GEE_WAIT_TIME = 0.1    # 请求间隔（秒）
    
    # 网格配置
    GRID_SPACING = 0.05    # 网格间距（度）
    GRID_BUFFER = 0.01     # 缓冲区（度）
    
    # 质量控制
    QA_MISSING_THRESHOLD = 0.3  # 缺失值阈值
    QA_FILL_METHOD = "interpolate"  # 填充方法
    
    # 支持的数据源
    SUPPORTED_SOURCES = [
        "LST",      # 地表温度
        "NDVI",     # 植被指数
        "PM25",     # PM2.5
        "ALBEDO",   # 反照率
        "EVI"       # 增强植被指数
    ]
    
    @classmethod
    def setup_directories(cls):
        """创建必要的目录"""
        for dir_path in [cls.DATA_DIR, cls.OUTPUT_DIR, cls.CACHE_DIR]:
            dir_path.mkdir(exist_ok=True)
        print(f"✅ 目录已创建")
    
    @classmethod
    def save_config(cls, filepath=None):
        """保存配置到JSON"""
        if filepath is None:
            filepath = cls.BASE_DIR / "config.json"
        
        config_data = {
            k: v for k, v in cls.__dict__.items() 
            if not k.startswith('_') and not callable(v)
        }
        
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(config_data, f, indent=2, default=str)
        
        print(f"✅ 配置已保存到: {filepath}")

# 初始化配置
Config.setup_directories()
Config.save_config()

print("\n📋 当前配置:")
print(f"  - 数据目录: {Config.DATA_DIR}")
print(f"  - 输出目录: {Config.OUTPUT_DIR}")
print(f"  - 缓存目录: {Config.CACHE_DIR}")
print(f"  - 支持的数据源: {', '.join(Config.SUPPORTED_SOURCES)}")

---

## 🌍 GEE数据提取器

完整的Google Earth Engine数据提取功能。

In [ ]:
# ============================================================# GEE数据提取器# ============================================================class GEEDataExtractor:    """    Google Earth Engine 数据提取器        支持提取多种遥感数据：    - LST (地表温度)    - NDVI (归一化植被指数)    - PM2.5 (颗粒物)    - Albedo (反照率)    - EVI (增强植被指数)    """        # 数据集配置    DATASETS = {        "LST": {            "collection": "MODIS/061/MOD11A2",            "band": "LST_Day_1km",            "scale": 1000,            "name": "地表温度"        },        "NDVI": {            "collection": "MODIS/061/MOD13Q1",            "band": "NDVI",            "scale": 250,            "name": "归一化植被指数"        },        "EVI": {            "collection": "MODIS/061/MOD13Q1",            "band": "EVI",            "scale": 250,            "name": "增强植被指数"        },        "ALBEDO": {            "collection": "MODIS/061/MCD43A3",            "band": "Albedo_WSA_shortwave",            "scale": 500,            "name": "反照率"        }    }        def __init__(self):        """初始化提取器"""        self._check_gee()        def _check_gee(self):        """检查GEE是否可用"""        try:            import ee            ee.Initialize()            self.ee = ee            self.gee_available = True            print("✅ GEE已初始化")        except Exception as e:            print(f"⚠️  GEE初始化失败: {e}")            self.gee_available = False        def extract_data(self, points_df, data_type, start_date, end_date,                     band_name=None, scale=None, reducer='mean'):        """        从GEE提取数据                参数:        -------        points_df : GeoDataFrame            包含几何信息的点数据，必须有geometry列        data_type : str            数据类型 (LST, NDVI, EVI, ALBEDO等)        start_date : str            开始日期 (YYYY-MM-DD)        end_date : str            结束日期 (YYYY-MM-DD)        band_name : str, optional            波段名称（如果不需要默认值）        scale : int, optional            分辨率（米）        reducer : str            聚合方法 ('mean', 'median', 'max', 'min')                返回:        -------        DataFrame with extracted values        """        if not self.gee_available:            raise RuntimeError("GEE不可用，请先认证: ee.Authenticate()")                # 获取数据集配置        if data_type not in self.DATASETS:            raise ValueError(f"不支持的数据类型: {data_type}")                dataset_config = self.DATASETS[data_type]        collection = dataset_config['collection']        band = band_name or dataset_config['band']        scale = scale or dataset_config['scale']                print(f"📥 正在提取 {dataset_config['name']} ({data_type})...")        print(f"   时间范围: {start_date} 到 {end_date}")        print(f"   数据点数: {len(points_df)}")                # 创建FeatureCollection        features = []        for idx, row in points_df.iterrows():            geom = row.geometry            point = self.ee.Geometry.Point([geom.x, geom.y])            feature = self.ee.Feature(point, {'id': idx})            features.append(feature)                fc = self.ee.FeatureCollection(features)                # 获取影像集合        image_collection = self.ee.ImageCollection(collection) \            .filterDate(start_date, end_date) \            .select(band)                # 选择聚合方法        if reducer == 'mean':            image_collection = image_collection.mean()        elif reducer == 'median':            image_collection = image_collection.median()        elif reducer == 'max':            image_collection = image_collection.max()        elif reducer == 'min':            image_collection = image_collection.min()                # 提取数据        try:            results = image_collection.sampleRegions(                collection=fc,                scale=scale,                geometries=True            )                        # 转换为DataFrame            data = results.getInfo()['features']                        extracted_data = []            for item in data:                props = item['properties']                extracted_data.append({                    'id': props.get('id'),                    data_type: props.get(band)                })                        result_df = pd.DataFrame(extracted_data)                        print(f"✅ 成功提取 {len(result_df)} 个数据点")                        return result_df                    except Exception as e:            print(f"❌ 提取失败: {e}")            return None        def extract_time_series(self, points_df, data_type,                            start_date, end_date,                            time_step='month'):        """        提取时间序列数据                参数:        -------        time_step : str            时间步长 ('month', 'week', 'day')                返回:        -------        DataFrame with time series        """        # 生成时间序列        dates = pd.date_range(start_date, end_date, freq='MS')                all_data = []                for i, date in enumerate(dates):            if i < len(dates) - 1:                period_end = dates[i + 1] - pd.Timedelta(days=1)            else:                period_end = pd.to_datetime(end_date)                        print(f"\n📅 处理时间段: {date.strftime('%Y-%m')}...")                        monthly_data = self.extract_data(                points_df=points_df,                data_type=data_type,                start_date=date.strftime('%Y-%m-%d'),                end_date=period_end.strftime('%Y-%m-%d')            )                        if monthly_data is not None:                monthly_data['date'] = date.strftime('%Y-%m')                all_data.append(monthly_data)                if all_data:            result_df = pd.concat(all_data, ignore_index=True)            print(f"\n✅ 时间序列提取完成，共 {len(result_df)} 条记录")            return result_df        else:            return None# 创建提取器实例extractor = GEEDataExtractor()print("\n✅ GEE数据提取器已准备就绪")

---

## 🏙️ 城市网格化管理

将城市区域划分为网格，方便批量提取数据。

In [ ]:
# ============================================================
# 城市网格化管理
# ============================================================

import shapely
from shapely.geometry import Point, Polygon, box

class CityGridManager:
    """
    城市网格化管理器
    
    功能：
    - 创建规则网格
    - 生成采样点
    - 管理城市边界
    """
    
    def __init__(self, spacing=Config.GRID_SPACING):
        """
        初始化
        
        参数:
        -------
        spacing : float
            网格间距（度）
        """
        self.spacing = spacing
        self.cities = {}
    
    def add_city_bbox(self, city_name, min_lon, min_lat, max_lon, max_lat):
        """
        添加城市边界框
        
        参数:
        -------
        city_name : str
            城市名称
        min_lon, min_lat, max_lon, max_lat : float
            边界坐标
        """
        self.cities[city_name] = {
            'bbox': (min_lon, min_lat, max_lon, max_lat),
            'polygon': box(min_lon, min_lat, max_lon, max_lat)
        }
        print(f"✅ 添加城市: {city_name}")
    
    def add_city_from_file(self, city_name, shapefile_path):
        """
        从Shapefile添加城市边界
        
        参数:
        -------
        city_name : str
            城市名称
        shapefile_path : str
            Shapefile路径
        """
        try:
            gdf = gpd.read_file(shapefile_path)
            # 合并所有几何
            unified = gdf.unary_union
            self.cities[city_name] = {
                'polygon': unified,
                'bbox': unified.bounds
            }
            print(f"✅ 从文件加载城市: {city_name}")
        except Exception as e:
            print(f"❌ 加载失败: {e}")
    
    def create_grid(self, city_name, buffer=Config.GRID_BUFFER):
        """
        为城市创建网格
        
        参数:
        -------
        city_name : str
            城市名称
        buffer : float
            缓冲距离（度）
        
        返回:
        -------
        GeoDataFrame
        

---

## 📊 示例：提取北京地表温度

下面是一个完整的示例，展示如何提取北京市的地表温度数据。

In [ ]:
# ============================================================
# 示例：提取北京地表温度
# ============================================================

# 第一步：定义北京边界
beijing_bbox = {
    'min_lon': 116.0,
    'min_lat': 39.7,
    'max_lon': 116.8,
    'max_lat': 40.2
}

print("📍 北京边界:")
print(f"   经度: {beijing_bbox['min_lon']}° ~ {beijing_bbox['max_lon']}°")
print(f"   纬度: {beijing_bbox['min_lat']}° ~ {beijing_bbox['max_lat']}°")

# 第二步：创建采样点
def create_sampling_points(bbox, spacing=0.05):
    """
    在边界框内创建规则采样点
    
    参数:
    -------
    bbox : dict
        边界框字典
    spacing : float
        点间距（度）
    
    返回:
    -------
    GeoDataFrame
    """
    lons = np.arange(bbox['min_lon'], bbox['max_lon'], spacing)
    lats = np.arange(bbox['min_lat'], bbox['max_lat'], spacing)
    
    points = []
    for lon in lons:
        for lat in lats:
            points.append({
                'longitude': lon,
                'latitude': lat,
                'geometry': Point(lon, lat)
            })
    
    return gpd.GeoDataFrame(points, crs='EPSG:4326')

# 创建采样点
beijing_points = create_sampling_points(beijing_bbox, spacing=0.05)
print(f"\n📍 创建了 {len(beijing_points)} 个采样点")

# 显示前几个点
print("\n前5个采样点:")
print(beijing_points.head())

In [ ]:
# 第三步：提取地表温度数据
# 注意：需要先认证GEE（如果还没有认证）

# 设置时间范围
start_date = '2023-01-01'
end_date = '2023-12-31'

print(f"\n📅 提取时间范围: {start_date} 到 {end_date}")
print(f"📊 数据类型: 地表温度 (LST)")

# 提取数据
try:
    lst_result = extractor.extract_data(
        points_df=beijing_points,
        data_type='LST',
        start_date=start_date,
        end_date=end_date,
        reducer='mean'
    )
    
    if lst_result is not None:
        print("\n✅ 数据提取成功！")
        print(f"   获得了 {len(lst_result)} 个数据点")
        
        # 显示统计信息
        if 'LST_Day_1km' in lst_result.columns:
            # 转换单位（MODIS LST需要缩放因子0.02）
            lst_result['LST_Celsius'] = lst_result['LST_Day_1km'] * 0.02 - 273.15
            
            print("\n📊 统计信息:")
            print(f"   平均温度: {lst_result['LST_Celsius'].mean():.2f}°C")
            print(f"   最高温度: {lst_result['LST_Celsius'].max():.2f}°C")
            print(f"   最低温度: {lst_result['LST_Celsius'].min():.2f}°C")
            print(f"   标准差: {lst_result['LST_Celsius'].std():.2f}°C")
            
            # 显示部分数据
            print("\n📋 数据预览:")
            print(lst_result[['LST_Celsius']].head())
            
            # 保存数据
            output_file = Config.OUTPUT_DIR / "beijing_lst_2023.csv"
            lst_result.to_csv(output_file, index=False)
            print(f"\n💾 数据已保存到: {output_file}")
    
except Exception as e:
    print(f"\n❌ 提取失败: {e}")
    print("\n💡 提示:")
    print("1. 确保已认证GEE: ee.Authenticate()")
    print("2. 检查网络连接")
    print("3. 查看错误信息")

---

## 📈 数据可视化

提取数据后，可以使用以下代码进行可视化。

In [ ]:
# ============================================================
# 数据可视化
# ============================================================

def plot_spatial_distribution(data, points, title="空间分布"):
    """
    绘制空间分布图
    
    参数:
    -------
    data : array-like
        要绘制的数据值
    points : GeoDataFrame
        采样点的几何信息
    title : str
        图表标题
    """
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # 绘制散点图
    scatter = ax.scatter(
        points.geometry.x,
        points.geometry.y,
        c=data,
        cmap='RdYlBu_r',
        s=50,
        alpha=0.7,
        edgecolors='black',
        linewidths=0.5
    )
    
    # 添加颜色条
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('数值', rotation=270, labelpad=20)
    
    ax.set_xlabel('经度', fontsize=12)
    ax.set_ylabel('纬度', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_histogram(data, title="数据分布", xlabel="数值"):
    """
    绘制直方图
    
    参数:
    -------
    data : array-like
        要绘制的数据
    title : str
        图表标题
    xlabel : str
        X轴标签
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.hist(data.dropna(), bins=30, edgecolor='black', alpha=0.7)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel('频数', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # 添加统计信息
    mean_val = data.mean()
    std_val = data.std()
    ax.axvline(mean_val, color='red', linestyle='--', 
               label=f'平均值: {mean_val:.2f}')
    ax.legend(fontsize=10)
    
    plt.tight_layout()
    plt.show()

# 示例：如果已提取数据，可以绘制
# plot_spatial_distribution(
#     lst_result['LST_Celsius'],
#     beijing_points,
#     title="北京地表温度分布 (2023年平均)"
# )

# plot_histogram(
#     lst_result['LST_Celsius'],
#     title="地表温度分布",
#     xlabel="温度 (°C)"
# )

print("✅ 可视化函数已加载")
print("\n使用示例:")
print("  plot_spatial_distribution(data, points, '标题')")
print("  plot_histogram(data, '标题', 'X轴标签')")

---

## 📦 批量处理

批量提取多个城市、多个时间段的数据。

In [ ]:
# ============================================================
# 批量处理
# ============================================================

class BatchProcessor:
    """
    批量处理器
    
    支持批量处理：
    - 多个城市
    - 多个时间段
    - 多种数据类型
    """
    
    def __init__(self, extractor):
        """
        初始化
        
        参数:
        -------
        extractor : GEEDataExtractor
            数据提取器实例
        """
        self.extractor = extractor
        self.results = {}
        self.tasks = []
    
    def add_task(self, city_name, points_df, data_type, 
                start_date, end_date, task_id=None):
        """
        添加提取任务
        
        参数:
        -------
        city_name : str
            城市名称
        points_df : GeoDataFrame
            采样点
        data_type : str
            数据类型
        start_date : str
            开始日期
        end_date : str
            结束日期
        task_id : str, optional
            任务ID
        """
        if task_id is None:
            task_id = f"{city_name}_{data_type}_{start_date}_{end_date}"
        
        self.tasks.append({
            'task_id': task_id,
            'city_name': city_name,
            'points_df': points_df,
            'data_type': data_type,
            'start_date': start_date,
            'end_date': end_date,
            'status': 'pending'
        })
        
        print(f"✅ 任务已添加: {task_id}")
    
    def run_all(self, save_results=True):
        """
        运行所有任务
        
        参数:
        -------
        save_results : bool
            是否保存结果
        
        返回:
        -------
        dict
            所有任务的结果
        """
        results = {}
        
        print(f"\n🚀 开始运行 {len(self.tasks)} 个任务...")
        
        for i, task in enumerate(self.tasks, 1):
            print(f"\n{'='*60}")
            print(f"任务 {i}/{len(self.tasks)}: {task['task_id']}")
            print(f"{'='*60}")
            
            try:
                result = self.extractor.extract_data(
                    points_df=task['points_df'],
                    data_type=task['data_type'],
                    start_date=task['start_date'],
                    end_date=task['end_date']
                )
                
                if result is not None:
                    results[task['task_id']] = result
                    task['status'] = 'completed'
                    
                    if save_results:
                        output_file = Config.OUTPUT_DIR / f"{task['task_id']}.csv"
                        result.to_csv(output_file, index=False)
                        print(f"💾 结果已保存: {output_file}")
                else:
                    task['status'] = 'failed'
                    
            except Exception as e:
                print(f"❌ 任务失败: {e}")
                task['status'] = 'error'
        
        self.results = results
        
        print(f"\n{'='*60}")
        print(f"✅ 批量处理完成")
        print(f"   成功: {sum(1 for t in self.tasks if t['status'] == 'completed')}")
        print(f"   失败: {sum(1 for t in self.tasks if t['status'] != 'completed')}")
        print(f"{'='*60}")
        
        return results
    
    def get_status_summary(self):
        """
        获取任务状态摘要
        
        返回:
        -------
        DataFrame
            任务状态表
        """
        return pd.DataFrame(self.tasks)

# 创建批量处理器
batch_processor = BatchProcessor(extractor)
print("✅ 批量处理器已创建")

---

## 🔧 工具函数

常用的工具函数集合。

In [ ]:
# ============================================================
# 工具函数
# ============================================================

def load_geojson(filepath):
    """
    加载GeoJSON文件
    
    参数:
    -------
    filepath : str
        文件路径
    
    返回:
    -------
    GeoDataFrame
    """
    return gpd.read_file(filepath)

def save_geojson(gdf, filepath):
    """
    保存为GeoJSON
    
    参数:
    -------
    gdf : GeoDataFrame
        要保存的数据
    filepath : str
        输出路径
    """
    gdf.to_file(filepath, driver='GeoJSON')
    print(f"✅ 已保存到: {filepath}")

def merge_datasets(datasets, on_column='id'):
    """
    合并多个数据集
    
    参数:
    -------
    datasets : list of DataFrame
        要合并的数据集列表
    on_column : str
        合并列
    
    返回:
    -------
    DataFrame
        合并后的数据集
    """
    result = datasets[0]
    for df in datasets[1:]:
        result = pd.merge(result, df, on=on_column, how='outer')
    return result

def quality_check(data, threshold=0.3):
    """
    数据质量检查
    
    参数:
    -------
    data : Series
        要检查的数据
    threshold : float
        缺失值阈值
    
    返回:
    -------
    dict
        质量报告
    """
    missing_ratio = data.isna().sum() / len(data)
    
    report = {
        'total_count': len(data),
        'valid_count': data.notna().sum(),
        'missing_count': data.isna().sum(),
        'missing_ratio': missing_ratio,
        'passed': missing_ratio < threshold
    }
    
    return report

def fill_missing(data, method='interpolate'):
    """
    填充缺失值
    
    参数:
    -------
    data : Series
        要填充的数据
    method : str
        填充方法 ('mean', 'median', 'interpolate', 'forward')
    
    返回:
    -------
    Series
        填充后的数据
    """
    if method == 'mean':
        return data.fillna(data.mean())
    elif method == 'median':
        return data.fillna(data.median())
    elif method == 'interpolate':
        return data.interpolate()
    elif method == 'forward':
        return data.fillna(method='ffill')
    else:
        return data

def export_report(data, output_path, title="数据报告"):
    """
    导出数据报告
    
    参数:
    -------
    data : DataFrame
        要报告的数据
    output_path : str
        输出路径
    title : str
        报告标题
    """
    report = f"""
# {title}
# 生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## 数据概览
- 数据量: {len(data)} 条
- 列数: {len(data.columns)} 列

## 列信息
{data.dtypes.to_string()}

## 统计摘要
{data.describe().to_string()}

## 缺失值统计
{data.isna().sum().to_string()}
"""
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(report)
    
    print(f"✅ 报告已保存: {output_path}")

print("✅ 工具函数已加载")
print("\n可用函数:")
print("  - load_geojson(filepath)")
print("  - save_geojson(gdf, filepath)")
print("  - merge_datasets(datasets)")
print("  - quality_check(data)")
print("  - fill_missing(data, method)")
print("  - export_report(data, output_path)")

---## 📚 快速参考### 常用操作流程#### 1. 提取单个城市、单个数据类型```python# 创建采样点points = create_sampling_points(bbox, spacing=0.05)# 提取数据result = extractor.extract_data(    points_df=points,    data_type='LST',    start_date='2023-01-01',    end_date='2023-12-31')```#### 2. 提取时间序列```pythonts_result = extractor.extract_time_series(    points_df=points,    data_type='LST',    start_date='2020-01-01',    end_date='2023-12-31',    time_step='month')```#### 3. 批量处理多个任务```python# 创建批量处理器processor = BatchProcessor(extractor)# 添加任务processor.add_task('北京', beijing_points, 'LST', '2023-01-01', '2023-12-31')processor.add_task('上海', shanghai_points, 'NDVI', '2023-01-01', '2023-12-31')# 运行所有任务results = processor.run_all()```### 支持的数据类型| 代码 | 名称 | 数据集 | 分辨率 ||------|------|--------|--------|| LST | 地表温度 | MODIS/061/MOD11A2 | 1000m || NDVI | 归一化植被指数 | MODIS/061/MOD13Q1 | 250m || EVI | 增强植被指数 | MODIS/061/MOD13Q1 | 250m || ALBEDO | 反照率 | MODIS/061/MCD43A3 | 500m |### 质量控制流程```python# 1. 检查数据质量report = quality_check(data['LST_Celsius'], threshold=0.3)# 2. 如果未通过，填充缺失值if not report['passed']:    data['LST_Celsius'] = fill_missing(data['LST_Celsius'], method='interpolate')# 3. 生成报告export_report(data, 'output/report.md', title='LST数据质量报告')```---## ❓ 常见问题### Q: 如何认证GEE？**A:** 运行以下命令：```pythonimport eeee.Authenticate()```### Q: 提取失败怎么办？**A:** 检查以下几点：1. 网络连接是否正常2. GEE是否已认证3. 时间范围是否合理4. 采样点数量是否过多（建议<5000个）### Q: 如何转换数据格式？**A:** 使用以下函数：```python# 保存为CSVdf.to_csv('output.csv', index=False)# 保存为Exceldf.to_excel('output.xlsx', index=False)# 保存为GeoJSONgdf.to_file('output.geojson', driver='GeoJSON')```---*Master Notebook - 最后更新: 2026-03-13*### 数据缩放说明不同数据类型的缩放因子：| 数据类型 | 缩放因子 | 公式 | 说明 ||---------|---------|------|------|| LST | 0.02 | `value × 0.02 - 273.15` | 转换为摄氏度 || NDVI | 0.0001 | `value × 0.0001` | 范围-1到1 || EVI | 0.0001 | `value × 0.0001` | 范围-1到1 || Albedo | 0.001 | `value × 0.001` | 0到1 |**示例**：```python# LST缩放df['LST_Celsius'] = df['LST_Day_1km'] * 0.02 - 273.15# NDVI缩放df['NDVI_scaled'] = df['NDVI'] * 0.0001# EVI缩放df['EVI_scaled'] = df['EVI'] * 0.0001# Albedo缩放df['Albedo_scaled'] = df['Albedo_WSA_shortwave'] * 0.001```